<a href="https://colab.research.google.com/github/sophyrise/fer-emotion-recognition/blob/main/fer2013_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FER2013 Facial Expression Recognition — Experiments Notebook

**Kaggle Competition:** Challenges in Representation Learning: FER2013  
**Task:** 7-class emotion classification from 48×48 grayscale face images  
**Classes:** Angry · Disgust · Fear · Happy · Sad · Surprise · Neutral

This notebook walks through all experiments:
1. Setup (GPU check, packages, data, WandB)
2. Dataset exploration
3. Sanity checks for all architectures
4. Arch 1: TinyMLP (underfitting baseline)
5. Arch 2: PlainCNN (overfitting demonstration)
6. Arch 3: RegCNN + hyperparameter search
7. Arch 4: MiniResNet + hyperparameter search
8. WandB automated sweeps
9. Results analysis

## 1. Setup


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Guard: every training run below MUST use the GPU. A silent CPU fallback is
# what killed the earlier `reg_no_aug` run (it logged nothing because it was
# too slow on CPU and got interrupted). Stop here if no GPU is attached.
assert torch.cuda.is_available(), (
    "No GPU detected. Set Runtime -> Change runtime type -> GPU (T4) and re-run, "
    "otherwise training will run on CPU and stall."
)

CUDA available: False


AssertionError: No GPU detected. Set Runtime -> Change runtime type -> GPU (T4) and re-run, otherwise training will run on CPU and stall.

In [ ]:
!pip install wandb scikit-learn -q

In [ ]:
import os
if os.path.exists('/content/fer'):
    !git -C /content/fer pull
else:
    !git clone https://github.com/sophyrise/fer-emotion-recognition.git /content/fer
%cd /content/fer

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!find /content/drive/MyDrive -name "icml_face_data.csv"

In [ ]:
!mkdir -p data
!cp "/content/drive/MyDrive/fer2013/icml_face_data.csv" data/icml_face_data.csv
!ls -lh data/


In [ ]:
import wandb
wandb.login()

ENTITY = "sgurj22-free-university-of-tbilisi-"
PROJECT = "fer2013"

## 2. Dataset Exploration

FER2013 contains 35,887 grayscale 48×48 face images split into:
- Training: 28,709 images
- PublicTest (val): 3,589 images  
- PrivateTest (test): 3,589 images

The dataset is **heavily imbalanced** — this is why we use Macro-F1 as the primary metric, not just accuracy.

In [ ]:
# Verify the dataset was copied in the setup section above.
import os
assert os.path.exists('../data/icml_face_data.csv'), "Run the setup cells first."
print("Dataset ready:", os.path.getsize('../data/icml_face_data.csv') // (1024 * 1024), "MB")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from dataset import EMOTIONS

df = pd.read_csv('../data/icml_face_data.csv')
df.columns = df.columns.str.strip()
print(df.head())
print(f"\nTotal samples: {len(df)}")
print(f"Splits: {df['Usage'].value_counts().to_dict()}")

In [ ]:

train_df = df[df['Usage'] == 'Training']
counts = train_df['emotion'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(EMOTIONS, counts.values, color='steelblue')
ax.set_title('Training set class distribution (FER2013)')
ax.set_ylabel('Number of samples')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nMax/Min ratio: {counts.max() / counts.min():.1f}x imbalance")

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for col, emotion_idx in enumerate(range(7)):
    subset = train_df[train_df['emotion'] == emotion_idx]
    for row in range(2):
        sample = subset.iloc[row]['pixels']
        img = np.array(sample.split(), dtype=np.uint8).reshape(48, 48)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(EMOTIONS[emotion_idx], fontsize=9)
plt.suptitle('Sample images per emotion class', y=1.02)
plt.tight_layout()
plt.show()

## 3. Sanity Checks

Before training any architecture we run three checks:

1. **Initial loss check**: At random init, CE loss should be ~ln(7) ≈ 1.946
2. **Overfit one batch**: Loss should reach ~0 in 200 steps (verifies training loop)
3. **Gradient flow**: All layers must have non-zero gradients (verifies backward pass)

These are equivalent to the forward-check and backward-check approach discussed in lectures.

In [ ]:
for arch in ['tiny', 'plain', 'reg', 'resnet']:
    print(f"\n{'='*60}")
    print(f"SANITY CHECKS — {arch.upper()}")
    print('='*60)
    !python src/sanity_checks.py --arch {arch}

## 4. Architecture 1: TinyMLP — Underfitting Baseline

```
Flatten → Linear(2304, 128) → ReLU → Linear(128, 7)
~297K parameters
```

**Design decision:** Start with the absolute minimum. No spatial inductive bias (treats pixels as independent), tiny hidden dimension. This will **underfit** — both training and validation accuracy will be low, and the gap between them will be small (both bad).

**Why this matters:** Establishes the floor and motivates CNNs.

In [ ]:
!python src/train.py \
    --arch tiny \
    --group arch_comparison \
    --name tiny_baseline \
    --epochs 30 \
    --lr 1e-3 \
    --batch 64 \
    --augment false \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run ea8dreu3 (0.0s)
wandb: ⣻ setting up run ea8dreu3 (0.0s)
wandb: ⣽ setting up run ea8dreu3 (0.0s)
wandb: ⣾ setting up run ea8dreu3 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_120647-ea8dreu3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tiny_baseline
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/ea8dreu3
Ep 00 | train 1.7049/0.3349 | val 1.6187/0.3739 | f1 0.2835 | gnorm 2.71
  -> Saved best (val_acc=0.3739)
Ep 01 | train 1.5665/0.3969 | v

## 5. Architecture 2: PlainCNN — Overfitting Demonstration

```
Conv(1→32)→ReLU→Pool → Conv(32→64)→ReLU→Pool → Conv(64→128)→ReLU→Pool
→ Linear(4608, 256) → ReLU → Linear(256, 7)
~1.2M parameters
```

**Design decision:** CNNs are the right architecture for images. But no regularization whatsoever — no BatchNorm, no Dropout, no augmentation. This will **overfit**: training accuracy will be high, but validation accuracy will plateau or decrease.

**Why this matters:** Demonstrates that capacity alone is not enough — regularization is essential.

In [ ]:
!python src/train.py \
    --arch plain \
    --group arch_comparison \
    --name plain_baseline \
    --epochs 30 \
    --lr 1e-3 \
    --batch 64 \
    --augment false \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run smg7drs3 (0.0s)
wandb: ⣻ setting up run smg7drs3 (0.0s)
wandb: ⣽ setting up run smg7drs3 (0.0s)
wandb: ⣾ setting up run smg7drs3 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_121833-smg7drs3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run plain_baseline
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/smg7drs3
Ep 00 | train 1.5386/0.3996 | val 1.3815/0.4712 | f1 0.3790 | gnorm 1.42
  -> Saved best (val_acc=0.4712)
Ep 01 | train 1.2684/0.5157 | 

## 6. Architecture 3: RegCNN — Regularized CNN

```
[Conv→BN→ReLU→Conv→BN→ReLU→Pool] × 3  (channels: 64→128→256)
→ Dropout(0.5) → Linear(9216, 512) → ReLU → Dropout(0.5) → Linear(512, 7)
~4.7M parameters
```

**Design decisions:**
- BatchNorm: stabilises training, allows higher LR, mild regularization
- Double conv per block: richer features before downsampling (VGG-style)
- Dropout (0.5): prevents neuron co-adaptation in FC layers
- Data augmentation: increases effective training set size

We explore several hyperparameter settings to understand what matters most.

In [ ]:
!python src/train.py \
    --arch reg \
    --group arch_comparison \
    --name reg_baseline \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment true \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run fffs48r3 (0.0s)
wandb: ⣻ setting up run fffs48r3 (0.0s)
wandb: ⣽ setting up run fffs48r3 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_123019-fffs48r3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reg_baseline
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/fffs48r3
Ep 00 | train 1.8946/0.2422 | val 1.7958/0.2499 | f1 0.0684 | gnorm 3.68
  -> Saved best (val_acc=0.2499)
Ep 01 | train 1.8084/0.2532 | val 1.7477/0.2616 | f1 0.0786 | gnorm 2.97

In [ ]:
!python src/train.py \
    --arch reg \
    --group reg_hp_search \
    --name reg_lr1e-4 \
    --epochs 40 --lr 1e-4 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment true \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run z2nihaw4 (0.0s)
wandb: ⣻ setting up run z2nihaw4 (0.0s)
wandb: ⣽ setting up run z2nihaw4 (0.0s)
wandb: ⣾ setting up run z2nihaw4 (0.0s)
wandb: ⣷ setting up run z2nihaw4 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_125516-z2nihaw4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reg_lr1e-4
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/z2nihaw4
Ep 00 | train 1.7578/0.2856 | val 1.5375/0.4115 | f1 0.2757 | gnorm 5.73
  -> Saved best (val_acc=0.

In [ ]:
!python src/train.py \
    --arch reg \
    --group reg_hp_search \
    --name reg_dropout06 \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.6 --wd 1e-4 --augment true \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run oqpb4mqk (0.0s)
wandb: ⣻ setting up run oqpb4mqk (0.0s)
wandb: ⣽ setting up run oqpb4mqk (0.0s)
wandb: ⣾ setting up run oqpb4mqk (0.0s)
wandb: ⣷ setting up run oqpb4mqk (0.5s)
wandb: ⣯ setting up run oqpb4mqk (0.5s)
wandb: ⣟ setting up run oqpb4mqk (0.5s)
wandb: ⡿ setting up run oqpb4mqk (0.5s)
wandb: ⢿ setting up run oqpb4mqk (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_132026-oqpb4mqk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reg_dropout06
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wand

In [12]:
!python src/train.py \
    --arch reg \
    --group reg_hp_search \
    --name reg_sgd_cosine \
    --epochs 40 --lr 1e-2 --batch 64 \
    --optimizer sgd --dropout 0.5 --wd 5e-4 --augment true \
    --sched cosine \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_144557-6hvf1gzy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reg_sgd_cosine
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/6hvf1gzy
Ep 00 | train 1.8363/0.2385 | val 1.8186/0.2494 | f1 0.0570 | gnorm 2.11
  -> Saved best (val_acc=0.2494)
Ep 01 | train 1.8101/0.2520 | val 1.7938/0.2544 | f1 0.0704 | gnorm 0.90
  -> Saved best (val_acc=0.2544)
Ep 02 | tr

In [13]:
!python src/train.py \
    --arch reg \
    --group reg_hp_search \
    --name reg_no_aug \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment false \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 3r8td0wa (0.0s)
wandb: ⣻ setting up run 3r8td0wa (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_151103-3r8td0wa
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reg_no_aug
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/3r8td0wa
Ep 00 | train 1.8459/0.2606 | val 1.7135/0.2990 | f1 0.1459 | gnorm 4.12
  -> Saved best (val_acc=0.2990)
Ep 01 | train 1.5898/0.3643 | val 1.4600/0.4218 | f1 0.2749 | gnorm 4.56
  -> Saved best (val_acc=0.4218)
Ep 02 | 

## 7. Architecture 4: MiniResNet — Residual Connections + Global Average Pooling

```
Stem: Conv(1→64)→BN→ReLU→Pool
Stage 1-3: StrideConv + ResBlock  (64→128→256→512)
Global Average Pooling → Dropout(0.4) → Linear(512, 7)
~5.3M parameters
```

**Key design decisions:**
- **Skip connections**: Gradient flows through identity path → no vanishing gradient in deeper net
- **Global Average Pooling**: Replaces the huge FC layer (9216→512 in RegCNN). GAP averages each feature map to one number → 512-dim vector, position-invariant, far fewer parameters
- **Label smoothing**: Prevents overconfident predictions, improves calibration

In [14]:
!python src/train.py \
    --arch resnet \
    --group arch_comparison \
    --name resnet_baseline \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched cosine --label_smooth 0.1 \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run aaowg4r9 (0.0s)
wandb: ⣻ setting up run aaowg4r9 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_153013-aaowg4r9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet_baseline
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/aaowg4r9
Ep 00 | train 1.7356/0.3310 | val 1.5950/0.4171 | f1 0.2994 | gnorm 2.24
  -> Saved best (val_acc=0.4171)
Ep 01 | train 1.5200/0.4612 | val 1.6900/0.4199 | f1 0.3016 | gnorm 1.89
  -> Saved best (val_acc=0.4199)
Ep 

In [15]:
!python src/train.py \
    --arch resnet \
    --group resnet_hp_search \
    --name resnet_bs32 \
    --epochs 40 --lr 5e-4 --batch 32 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched cosine \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_161521-7v2pqmp6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet_bs32
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/7v2pqmp6
Ep 00 | train 1.6836/0.3283 | val 1.4624/0.4372 | f1 0.3511 | gnorm 3.53
  -> Saved best (val_acc=0.4372)
Ep 01 | train 1.3929/0.4641 | val 1.3078/0.5004 | f1 0.4035 | gnorm 3.20
  -> Save

In [13]:
# Ablation: same as resnet_baseline but WITHOUT label smoothing.
# Isolates the effect of label_smooth=0.1 on calibration / val_acc.
!python src/train.py \
    --arch resnet \
    --group resnet_hp_search \
    --name resnet_no_smooth \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched cosine --label_smooth 0.0 \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run n4ulnrs7 (0.0s)
wandb: ⣻ setting up run n4ulnrs7 (0.0s)
wandb: ⣽ setting up run n4ulnrs7 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_174433-n4ulnrs7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet_no_smooth
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/n4ulnrs7
Ep 00 | train 1.6756/0.3249 | val 1.5744/0.3842 | f1 0.3263 | gnorm 2.57
  -> Saved best (val_acc=0.3842)
Ep 01 | train 1.3858/0.4699 | val 1.2943/0.4996 | f1 0.3787 | gnorm 

In [14]:
!python src/train.py \
    --arch resnet \
    --group resnet_hp_search \
    --name resnet_plateau \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched plateau --label_smooth 0.1 \
    --workers 2

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (sgurj22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 1cjppld4 (0.0s)
wandb: ⣻ setting up run 1cjppld4 (0.0s)
wandb: ⣽ setting up run 1cjppld4 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/fer/wandb/run-20260614_180641-1cjppld4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet_plateau
wandb: ⭐️ View project at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013
wandb: 🚀 View run at https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/runs/1cjppld4
Ep 00 | train 1.7522/0.3167 | val 1.5482/0.4472 | f1 0.3211 | gnorm 2.25
  -> Saved best (val_acc=0.4472)
Ep 01 | train 1.5274/0.4536 | val 1.4624/0.4932 | f1 0.3790 | gnorm 1.

## 8. WandB Automated Bayesian Sweeps

Beyond manual HP search, we run automated Bayesian optimization sweeps. The sweep agent samples HP combinations and focuses on the most promising regions of the search space.

In [17]:
!git -C /content/fer pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 461 bytes | 461.00 KiB/s, done.
From https://github.com/sophyrise/fer-emotion-recognition
   036969a..2271b82  main       -> origin/main
Updating 036969a..2271b82
Fast-forward
 configs/sweep.yaml        | 1 +
 configs/sweep_resnet.yaml | 1 +
 2 files changed, 2 insertions(+)


In [18]:
!wandb sweep configs/sweep.yaml

wandb: Creating sweep from: configs/sweep.yaml
wandb: Creating sweep with ID: qh4acgv2
wandb: View sweep at: https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/sweeps/qh4acgv2
wandb: Run sweep agent with: wandb agent sgurj22-free-university-of-tbilisi-/fer2013/qh4acgv2


In [ ]:
REG_SWEEP_ID = "qh4acgv2"

!wandb agent {ENTITY}/{PROJECT}/{REG_SWEEP_ID} --count 6

wandb: Starting wandb agent 🕵️
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
2026-06-14 18:40:20,345 - wandb.wandb_agent - INFO - Running runs: []
2026-06-14 18:40:21,010 - wandb.wandb_agent - INFO - Agent received command: run
2026-06-14 18:40:21,011 - wandb.wandb_agent - INFO - Agent starting run with config:
	arch: reg
	augment: true
	batch: 32
	csv: data/icml_face_data.csv
	dropout: 0.6
	epochs: 40
	group: reg_sweep
	label_smooth: 0
	lr: 0.00658975306501457
	name: sweep_run
	optimizer: adam
	sched: plateau
	wd: 0
2026-06-14 18:40:21,012 - wandb.wandb_agent - INFO - About to run command: /usr/bin/env python src/train.py --arch=reg --augment=true --batch=32 --csv=data/icml_face_data.csv --dropout=0.6 --epochs=40 --group=reg_sweep --label_smooth=0 --lr=0.00658975306501457 --name=sweep_run --optimizer=adam --sched=plateau --wd=0
Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
2026-06-14 18:40:26,

In [12]:
!git -C /content/fer pull

Already up to date.


In [13]:
!wandb sweep configs/sweep_resnet.yaml

wandb: Creating sweep from: configs/sweep_resnet.yaml
wandb: Creating sweep with ID: r2rf17is
wandb: View sweep at: https://wandb.ai/sgurj22-free-university-of-tbilisi-/fer2013/sweeps/r2rf17is
wandb: Run sweep agent with: wandb agent sgurj22-free-university-of-tbilisi-/fer2013/r2rf17is


In [14]:
# Copy the sweep id printed by the cell above and paste it here.
RESNET_SWEEP_ID = "r2rf17is"

!wandb agent {ENTITY}/{PROJECT}/{RESNET_SWEEP_ID} --count 12

wandb: Starting wandb agent 🕵️
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
2026-06-15 12:49:30,266 - wandb.wandb_agent - INFO - Running runs: []
2026-06-15 12:49:31,442 - wandb.wandb_agent - INFO - Agent received command: run
2026-06-15 12:49:31,443 - wandb.wandb_agent - INFO - Agent starting run with config:
	arch: resnet
	augment: true
	batch: 64
	csv: data/icml_face_data.csv
	dropout: 0.5
	epochs: 20
	group: resnet_sweep
	label_smooth: 0.1
	lr: 0.003294644410731979
	name: sweep_run
	optimizer: adam
	sched: plateau
	wd: 0.0005
2026-06-15 12:49:31,444 - wandb.wandb_agent - INFO - About to run command: /usr/bin/env python src/train.py --arch=resnet --augment=true --batch=64 --csv=data/icml_face_data.csv --dropout=0.5 --epochs=20 --group=resnet_sweep --label_smooth=0.1 --lr=0.003294644410731979 --name=sweep_run --optimizer=adam --sched=plateau --wd=0.0005
Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/

In [ ]:
!git -C /content/fer pull

In [ ]:
!python src/train.py \
    --arch reg \
    --group reg_hp_search \
    --name reg_weighted \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment true \
    --class_weights true \
    --workers 2

## 9. Results Analysis

After all runs complete, compare architectures directly in WandB:
- Go to your WandB project → Group by `arch` or filter by `group=arch_comparison`
- Compare `val_acc`, `f1_macro`, and `val_loss` curves side by side
- Look at the `conf_mat` to see which emotions are confused with each other

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

api = wandb.Api()

runs = api.runs(f"{ENTITY}/{PROJECT}", filters={"group": "arch_comparison"})

results = []
for run in runs:
    summary = run.summary
    results.append({
        'name': run.name,
        'arch': run.config.get('arch'),
        'val_acc': summary.get('val_acc'),
        'test_acc': summary.get('test_acc'),
        'f1_macro': summary.get('f1_macro'),
        'test_f1': summary.get('test_f1'),
    })

results_df = pd.DataFrame(results).sort_values('val_acc', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(results_df['arch'], results_df['test_acc'], color='steelblue')
    axes[0].set_title('Test Accuracy by Architecture')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1)
    for i, v in enumerate(results_df['test_acc']):
        if v:
            axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

    axes[1].bar(results_df['arch'], results_df['test_f1'], color='darkorange')
    axes[1].set_title('Test Macro-F1 by Architecture')
    axes[1].set_ylabel('Macro F1')
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(results_df['test_f1']):
        if v:
            axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center')

    plt.tight_layout()
    plt.show()

## Key Takeaways

| Architecture | Behaviour | Root Cause | Fix Applied |
|---|---|---|---|
| TinyMLP | **Underfitting** | No spatial inductive bias, 128 hidden units too small | → Switch to CNN |
| PlainCNN | **Overfitting** | No regularisation, large FC memorises training set | → Add BN, Dropout, augmentation |
| RegCNN | **Good fit** | Regularisation balances capacity and generalisation | → Add residual connections, GAP |
| MiniResNet | **Best** | Skip connections + GAP = deeper + fewer params + position-invariant | Final model |

**On class imbalance:** FER2013 has a 13:1 imbalance (Happy vs Disgust). Macro-F1 reveals this — architectures with high accuracy but low F1 are ignoring minority classes.

**On augmentation:** The ablation run (`reg_no_aug`) shows how much augmentation contributes. Removing it causes val_acc to drop significantly, confirming that data augmentation is not optional for this dataset.